# CRISP-DM Phase 4: Modeling

This notebook reads the Phase 3 prepared crash table, trains the Phase 4 severity-risk model, evaluates it on 2025, and writes the artifacts to generated/modeling/


## Setup

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


MODELING_START_YEAR = 2018
TRAIN_END_YEAR = 2024
TEST_YEAR = 2025

GRID_LAT_ORIGIN = 41.60
GRID_LON_ORIGIN = -87.95
GRID_LAT_STEP = 0.005
GRID_LON_STEP = 0.006

ZERO_SAMPLE_SIZE = 750_000
RANDOM_STATE = 42
SCENARIO_NAMES = ["overall", "rain", "dark", "wet_surface", "rain_dark_wet", "pedbike"]
MODEL_HYPERPARAMETERS = {
    "loss": "squared_error",
    "learning_rate": 0.05,
    "max_iter": 250,
    "max_depth": 6,
    "min_samples_leaf": 100,
    "l2_regularization": 0.1,
    "random_state": RANDOM_STATE,
}

PREPARED_USECOLS = [
    "CRASH_RECORD_ID",
    "crash_ts",
    "crash_year",
    "crash_month",
    "crash_hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "is_rush_hour",
    "grid_id",
    "grid_center_lat",
    "grid_center_lon",
    "POSTED_SPEED_LIMIT",
    "is_rain",
    "is_dark",
    "is_wet_surface",
    "is_intersection",
    "is_work_zone",
    "is_pedbike",
    "injury_crash",
    "serious_injury_crash",
    "fatal_crash",
    "severity_weight",
]

FEATURE_COLUMNS = [
    "grid_center_lat",
    "grid_center_lon",
    "crash_month",
    "crash_hour",
    "is_weekend",
    "is_night",
    "is_rush_hour",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "grid_mean_target",
    "grid_nonzero_slot_share",
    "grid_time_mean_target",
    "grid_month_mean_target",
    "grid_hour_mean_target",
    "grid_weekend_mean_target",
    "global_mean_target",
    "global_time_mean_target",
    "global_month_mean_target",
    "global_hour_mean_target",
    "global_weekend_mean_target",
    "train_crash_total",
    "train_severity_per_crash",
    "avg_speed_limit",
    "rain_share",
    "dark_share",
    "wet_surface_share",
    "intersection_share",
    "work_zone_share",
    "pedbike_share",
    "injury_share",
    "serious_injury_share",
    "fatal_share",
]


## Paths And Loading Helpers

In [2]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "crisp_stages").exists() and (candidate / "README.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root")

ROOT = find_repo_root(Path.cwd())
PREPARED_DIR = ROOT / "generated" / "prepared"
OUTPUT_DIR = ROOT / "generated" / "modeling"
PREPARED_CRASHES_PATH = PREPARED_DIR / "prepared_crash_features.csv.gz"


## Data Engineering

In [3]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(math.sqrt(mean_squared_error(y_true, y_pred)))

def load_prepared_crashes() -> pd.DataFrame:
    if not PREPARED_CRASHES_PATH.exists():
        raise FileNotFoundError(
            "Missing generated/prepared/prepared_crash_features.csv.gz. "
            "Run the Phase 3 data preparation notebook before Phase 4."
        )

    crashes = pd.read_csv(PREPARED_CRASHES_PATH, usecols=PREPARED_USECOLS, low_memory=False)
    crashes["crash_ts"] = pd.to_datetime(crashes["crash_ts"], errors="coerce")
    crashes = crashes[crashes["crash_ts"].notna()].copy()
    crashes = crashes[crashes["crash_year"].between(MODELING_START_YEAR, TEST_YEAR)].copy()

    numeric_cols = [
        "crash_year",
        "crash_month",
        "crash_hour",
        "day_of_week",
        "is_weekend",
        "is_night",
        "is_rush_hour",
        "POSTED_SPEED_LIMIT",
        "grid_center_lat",
        "grid_center_lon",
        "is_rain",
        "is_dark",
        "is_wet_surface",
        "is_intersection",
        "is_work_zone",
        "is_pedbike",
        "injury_crash",
        "serious_injury_crash",
        "fatal_crash",
        "severity_weight",
    ]
    for col in numeric_cols:
        crashes[col] = pd.to_numeric(crashes[col], errors="coerce").fillna(0)

    return crashes.sort_values(["crash_ts", "CRASH_RECORD_ID"]).reset_index(drop=True)


## Grid-Time Panel Construction

In [4]:
def build_grid_lookup(crashes: pd.DataFrame) -> pd.DataFrame:
    lookup = (
        crashes[["grid_id", "grid_center_lat", "grid_center_lon"]]
        .drop_duplicates()
        .sort_values("grid_id")
        .reset_index(drop=True)
    )
    lookup["grid_code"] = np.arange(len(lookup), dtype=np.int32)
    return lookup[["grid_code", "grid_id", "grid_center_lat", "grid_center_lon"]]

def build_positive_panel(crashes: pd.DataFrame, grid_lookup: pd.DataFrame) -> pd.DataFrame:
    crashes = crashes.merge(grid_lookup[["grid_id", "grid_code"]], on="grid_id", how="left", validate="m:1")
    panel = (
        crashes.groupby(
            [
                "grid_code",
                "grid_id",
                "grid_center_lat",
                "grid_center_lon",
                "crash_year",
                "crash_month",
                "crash_hour",
                "is_weekend",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            crash_count=("CRASH_RECORD_ID", "size"),
            injury_crashes=("injury_crash", "sum"),
            serious_injury_crashes=("serious_injury_crash", "sum"),
            fatal_crashes=("fatal_crash", "sum"),
            severity_weight_sum=("severity_weight", "sum"),
        )
        .sort_values(["crash_year", "crash_month", "crash_hour", "grid_code", "is_weekend"])
        .reset_index(drop=True)
    )
    return panel

def encode_panel_codes(
    grid_code: np.ndarray,
    crash_year: np.ndarray,
    crash_month: np.ndarray,
    crash_hour: np.ndarray,
    is_weekend: np.ndarray,
    base_year: int,
    year_count: int,
) -> np.ndarray:
    year_offset = crash_year.astype(np.int64) - base_year
    code = grid_code.astype(np.int64)
    code = code * year_count + year_offset
    code = code * 12 + (crash_month.astype(np.int64) - 1)
    code = code * 24 + crash_hour.astype(np.int64)
    code = code * 2 + is_weekend.astype(np.int64)
    return code

def decode_panel_codes(codes: np.ndarray, grid_lookup: pd.DataFrame, base_year: int, year_count: int) -> pd.DataFrame:
    values = codes.astype(np.int64).copy()
    is_weekend = (values % 2).astype(np.int8)
    values //= 2
    crash_hour = (values % 24).astype(np.int8)
    values //= 24
    crash_month = (values % 12 + 1).astype(np.int8)
    values //= 12
    crash_year = (base_year + (values % year_count)).astype(np.int16)
    grid_code = (values // year_count).astype(np.int32)

    grid_ids = grid_lookup["grid_id"].to_numpy()
    grid_lats = grid_lookup["grid_center_lat"].to_numpy()
    grid_lons = grid_lookup["grid_center_lon"].to_numpy()

    frame = pd.DataFrame(
        {
            "grid_code": grid_code,
            "grid_id": grid_ids[grid_code],
            "grid_center_lat": grid_lats[grid_code],
            "grid_center_lon": grid_lons[grid_code],
            "crash_year": crash_year,
            "crash_month": crash_month,
            "crash_hour": crash_hour,
            "is_weekend": is_weekend,
        }
    )
    frame["crash_count"] = 0
    frame["injury_crashes"] = 0
    frame["serious_injury_crashes"] = 0
    frame["fatal_crashes"] = 0
    frame["severity_weight_sum"] = 0.0
    return frame

def sample_training_zero_panel(train_positive: pd.DataFrame, grid_lookup: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    year_count = TRAIN_END_YEAR - MODELING_START_YEAR + 1
    total_space = len(grid_lookup) * year_count * 12 * 24 * 2

    positive_codes = encode_panel_codes(
        train_positive["grid_code"].to_numpy(),
        train_positive["crash_year"].to_numpy(),
        train_positive["crash_month"].to_numpy(),
        train_positive["crash_hour"].to_numpy(),
        train_positive["is_weekend"].to_numpy(),
        base_year=MODELING_START_YEAR,
        year_count=year_count,
    )

    occupied = np.zeros(total_space, dtype=bool)
    occupied[positive_codes] = True
    zero_codes = np.flatnonzero(~occupied)

    rng = np.random.default_rng(RANDOM_STATE)
    sample_size = min(ZERO_SAMPLE_SIZE, len(zero_codes))
    sampled_zero_codes = rng.choice(zero_codes, size=sample_size, replace=False)
    sampled_zero = decode_panel_codes(sampled_zero_codes, grid_lookup, MODELING_START_YEAR, year_count)
    sampled_zero["sample_weight"] = float(len(zero_codes) / sample_size)

    metadata = {
        "train_panel_total_rows": int(total_space),
        "train_positive_rows": int(len(train_positive)),
        "train_zero_rows_total": int(len(zero_codes)),
        "train_zero_rows_sampled": int(sample_size),
        "sampled_zero_weight": float(len(zero_codes) / sample_size),
    }
    return sampled_zero, metadata

def build_full_test_panel(grid_lookup: pd.DataFrame, test_positive: pd.DataFrame) -> pd.DataFrame:
    grids = grid_lookup["grid_code"].to_numpy(dtype=np.int32)
    grid_ids = grid_lookup["grid_id"].to_numpy()
    grid_lats = grid_lookup["grid_center_lat"].to_numpy()
    grid_lons = grid_lookup["grid_center_lon"].to_numpy()

    month_values = np.arange(1, 13, dtype=np.int8)
    hour_values = np.arange(24, dtype=np.int8)
    weekend_values = np.array([0, 1], dtype=np.int8)

    grid_code = np.repeat(grids, len(month_values) * len(hour_values) * len(weekend_values))
    crash_month = np.tile(np.repeat(month_values, len(hour_values) * len(weekend_values)), len(grids))
    crash_hour = np.tile(np.repeat(hour_values, len(weekend_values)), len(grids) * len(month_values))
    is_weekend = np.tile(weekend_values, len(grids) * len(month_values) * len(hour_values))

    test_panel = pd.DataFrame(
        {
            "grid_code": grid_code,
            "grid_id": grid_ids[grid_code],
            "grid_center_lat": grid_lats[grid_code],
            "grid_center_lon": grid_lons[grid_code],
            "crash_year": TEST_YEAR,
            "crash_month": crash_month,
            "crash_hour": crash_hour,
            "is_weekend": is_weekend,
        }
    )

    test_panel = test_panel.merge(
        test_positive[
            [
                "grid_code",
                "crash_year",
                "crash_month",
                "crash_hour",
                "is_weekend",
                "crash_count",
                "injury_crashes",
                "serious_injury_crashes",
                "fatal_crashes",
                "severity_weight_sum",
            ]
        ],
        on=["grid_code", "crash_year", "crash_month", "crash_hour", "is_weekend"],
        how="left",
        validate="1:1",
    )
    fill_cols = ["crash_count", "injury_crashes", "serious_injury_crashes", "fatal_crashes", "severity_weight_sum"]
    for col in fill_cols:
        test_panel[col] = test_panel[col].fillna(0)
    return test_panel


## Feature Tables And Model Training

In [5]:
def build_feature_tables(train_crashes: pd.DataFrame, train_positive: pd.DataFrame, grid_lookup: pd.DataFrame) -> dict:
    n_grids = len(grid_lookup)
    n_years = TRAIN_END_YEAR - MODELING_START_YEAR + 1
    slots_per_grid = n_years * 12 * 24 * 2
    slots_per_grid_month = n_years * 24 * 2
    slots_per_grid_hour = n_years * 12 * 2
    slots_per_grid_weekend = n_years * 12 * 24
    slots_per_global_month = n_grids * slots_per_grid_month
    slots_per_global_hour = n_grids * slots_per_grid_hour
    slots_per_global_weekend = n_grids * slots_per_grid_weekend
    slots_per_global_time = n_grids * n_years

    grid_profile = (
        train_crashes.groupby(["grid_id", "grid_center_lat", "grid_center_lon"], as_index=False)
        .agg(
            train_crash_total=("CRASH_RECORD_ID", "size"),
            train_severity_sum=("severity_weight", "sum"),
            train_severity_per_crash=("severity_weight", "mean"),
            avg_speed_limit=("POSTED_SPEED_LIMIT", "mean"),
            rain_share=("is_rain", "mean"),
            dark_share=("is_dark", "mean"),
            wet_surface_share=("is_wet_surface", "mean"),
            intersection_share=("is_intersection", "mean"),
            work_zone_share=("is_work_zone", "mean"),
            pedbike_share=("is_pedbike", "mean"),
            injury_share=("injury_crash", "mean"),
            serious_injury_share=("serious_injury_crash", "mean"),
            fatal_share=("fatal_crash", "mean"),
        )
    )
    grid_nonzero = train_positive.groupby("grid_id", as_index=False).agg(nonzero_slots=("severity_weight_sum", "size"))
    grid_profile = grid_profile.merge(grid_nonzero, on="grid_id", how="left", validate="1:1")
    grid_profile["nonzero_slots"] = grid_profile["nonzero_slots"].fillna(0)
    grid_profile["grid_nonzero_slot_share"] = grid_profile["nonzero_slots"] / slots_per_grid
    grid_profile["grid_mean_target"] = grid_profile["train_severity_sum"] / slots_per_grid
    grid_profile = grid_profile.drop(columns=["train_severity_sum", "nonzero_slots"])

    def mean_table(keys: list[str], value_name: str, denominator: int) -> pd.DataFrame:
        table = train_positive.groupby(keys, as_index=False).agg(severity_weight_sum=("severity_weight_sum", "sum"))
        table[value_name] = table["severity_weight_sum"] / denominator
        return table.drop(columns=["severity_weight_sum"])

    grid_time_mean = mean_table(["grid_id", "crash_month", "crash_hour", "is_weekend"], "grid_time_mean_target", n_years)
    grid_month_mean = mean_table(["grid_id", "crash_month"], "grid_month_mean_target", slots_per_grid_month)
    grid_hour_mean = mean_table(["grid_id", "crash_hour"], "grid_hour_mean_target", slots_per_grid_hour)
    grid_weekend_mean = mean_table(["grid_id", "is_weekend"], "grid_weekend_mean_target", slots_per_grid_weekend)
    global_time_mean = mean_table(["crash_month", "crash_hour", "is_weekend"], "global_time_mean_target", slots_per_global_time)
    global_month_mean = mean_table(["crash_month"], "global_month_mean_target", slots_per_global_month)
    global_hour_mean = mean_table(["crash_hour"], "global_hour_mean_target", slots_per_global_hour)
    global_weekend_mean = mean_table(["is_weekend"], "global_weekend_mean_target", slots_per_global_weekend)

    global_defaults = {
        "global_mean_target": float(train_positive["severity_weight_sum"].sum() / (n_grids * slots_per_grid)),
        "avg_speed_limit": float(train_crashes["POSTED_SPEED_LIMIT"].mean()),
        "rain_share": float(train_crashes["is_rain"].mean()),
        "dark_share": float(train_crashes["is_dark"].mean()),
        "wet_surface_share": float(train_crashes["is_wet_surface"].mean()),
        "intersection_share": float(train_crashes["is_intersection"].mean()),
        "work_zone_share": float(train_crashes["is_work_zone"].mean()),
        "pedbike_share": float(train_crashes["is_pedbike"].mean()),
        "injury_share": float(train_crashes["injury_crash"].mean()),
        "serious_injury_share": float(train_crashes["serious_injury_crash"].mean()),
        "fatal_share": float(train_crashes["fatal_crash"].mean()),
        "train_crash_total": float(grid_profile["train_crash_total"].mean()),
        "train_severity_per_crash": float(train_crashes["severity_weight"].mean()),
        "grid_nonzero_slot_share": float(grid_profile["grid_nonzero_slot_share"].mean()),
    }

    return {
        "grid_profile": grid_profile,
        "grid_time_mean": grid_time_mean,
        "grid_month_mean": grid_month_mean,
        "grid_hour_mean": grid_hour_mean,
        "grid_weekend_mean": grid_weekend_mean,
        "global_time_mean": global_time_mean,
        "global_month_mean": global_month_mean,
        "global_hour_mean": global_hour_mean,
        "global_weekend_mean": global_weekend_mean,
        "global_defaults": global_defaults,
    }

def add_model_features(panel: pd.DataFrame, feature_tables: dict) -> pd.DataFrame:
    frame = panel.copy()
    frame = frame.merge(feature_tables["grid_profile"], on=["grid_id", "grid_center_lat", "grid_center_lon"], how="left")
    frame = frame.merge(feature_tables["grid_time_mean"], on=["grid_id", "crash_month", "crash_hour", "is_weekend"], how="left")
    frame = frame.merge(feature_tables["grid_month_mean"], on=["grid_id", "crash_month"], how="left")
    frame = frame.merge(feature_tables["grid_hour_mean"], on=["grid_id", "crash_hour"], how="left")
    frame = frame.merge(feature_tables["grid_weekend_mean"], on=["grid_id", "is_weekend"], how="left")
    frame = frame.merge(feature_tables["global_time_mean"], on=["crash_month", "crash_hour", "is_weekend"], how="left")
    frame = frame.merge(feature_tables["global_month_mean"], on=["crash_month"], how="left")
    frame = frame.merge(feature_tables["global_hour_mean"], on=["crash_hour"], how="left")
    frame = frame.merge(feature_tables["global_weekend_mean"], on=["is_weekend"], how="left")

    frame["global_mean_target"] = feature_tables["global_defaults"]["global_mean_target"]
    frame["is_night"] = frame["crash_hour"].isin([20, 21, 22, 23, 0, 1, 2, 3, 4, 5]).astype("int8")
    frame["is_rush_hour"] = (
        (frame["is_weekend"] == 0) & frame["crash_hour"].isin([7, 8, 9, 16, 17, 18])
    ).astype("int8")
    frame["month_sin"] = np.sin(2 * np.pi * frame["crash_month"] / 12.0)
    frame["month_cos"] = np.cos(2 * np.pi * frame["crash_month"] / 12.0)
    frame["hour_sin"] = np.sin(2 * np.pi * frame["crash_hour"] / 24.0)
    frame["hour_cos"] = np.cos(2 * np.pi * frame["crash_hour"] / 24.0)

    defaults = feature_tables["global_defaults"]
    fill_map = {
        "grid_mean_target": defaults["global_mean_target"],
        "grid_nonzero_slot_share": defaults["grid_nonzero_slot_share"],
        "grid_time_mean_target": 0.0,
        "grid_month_mean_target": 0.0,
        "grid_hour_mean_target": 0.0,
        "grid_weekend_mean_target": 0.0,
        "global_time_mean_target": 0.0,
        "global_month_mean_target": 0.0,
        "global_hour_mean_target": 0.0,
        "global_weekend_mean_target": 0.0,
        "train_crash_total": defaults["train_crash_total"],
        "train_severity_per_crash": defaults["train_severity_per_crash"],
        "avg_speed_limit": defaults["avg_speed_limit"],
        "rain_share": defaults["rain_share"],
        "dark_share": defaults["dark_share"],
        "wet_surface_share": defaults["wet_surface_share"],
        "intersection_share": defaults["intersection_share"],
        "work_zone_share": defaults["work_zone_share"],
        "pedbike_share": defaults["pedbike_share"],
        "injury_share": defaults["injury_share"],
        "serious_injury_share": defaults["serious_injury_share"],
        "fatal_share": defaults["fatal_share"],
    }
    for col, value in fill_map.items():
        frame[col] = frame[col].fillna(value)

    return frame

def train_model(train_frame: pd.DataFrame) -> HistGradientBoostingRegressor:
    model = HistGradientBoostingRegressor(**MODEL_HYPERPARAMETERS)
    model.fit(
        train_frame[FEATURE_COLUMNS],
        np.log1p(train_frame["severity_weight_sum"]),
        sample_weight=train_frame["sample_weight"],
    )
    return model

def evaluate_predictions(test_frame: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    actual = test_frame["severity_weight_sum"].to_numpy()
    baseline = test_frame["baseline_prediction"].to_numpy()
    predicted = test_frame["model_prediction"].to_numpy()

    grid_eval = (
        test_frame.groupby(["grid_id", "grid_center_lat", "grid_center_lon"], as_index=False)
        .agg(
            actual_severity_sum=("severity_weight_sum", "sum"),
            baseline_severity_sum=("baseline_prediction", "sum"),
            predicted_severity_sum=("model_prediction", "sum"),
            actual_crash_count=("crash_count", "sum"),
        )
        .sort_values("predicted_severity_sum", ascending=False)
        .reset_index(drop=True)
    )

    top_k = 100
    actual_top = set(grid_eval.nlargest(top_k, "actual_severity_sum")["grid_id"])
    model_top = set(grid_eval.nlargest(top_k, "predicted_severity_sum")["grid_id"])
    baseline_top = set(grid_eval.nlargest(top_k, "baseline_severity_sum")["grid_id"])

    metrics = {
        "row_level": {
            "baseline_mae": float(mean_absolute_error(actual, baseline)),
            "baseline_rmse": rmse(actual, baseline),
            "model_mae": float(mean_absolute_error(actual, predicted)),
            "model_rmse": rmse(actual, predicted),
        },
        "grid_level": {
            "baseline_mae": float(mean_absolute_error(grid_eval["actual_severity_sum"], grid_eval["baseline_severity_sum"])),
            "baseline_rmse": rmse(
                grid_eval["actual_severity_sum"].to_numpy(),
                grid_eval["baseline_severity_sum"].to_numpy(),
            ),
            "model_mae": float(mean_absolute_error(grid_eval["actual_severity_sum"], grid_eval["predicted_severity_sum"])),
            "model_rmse": rmse(
                grid_eval["actual_severity_sum"].to_numpy(),
                grid_eval["predicted_severity_sum"].to_numpy(),
            ),
            "model_r2": float(r2_score(grid_eval["actual_severity_sum"], grid_eval["predicted_severity_sum"])),
            "top_100_overlap_model": float(len(actual_top & model_top) / top_k),
            "top_100_overlap_baseline": float(len(actual_top & baseline_top) / top_k),
        },
    }
    return metrics, grid_eval

def build_prediction_scores(test_frame: pd.DataFrame, grid_eval: pd.DataFrame) -> tuple[float, float]:
    row_reference = float(np.quantile(test_frame["model_prediction"], 0.99))
    grid_reference = float(np.quantile(grid_eval["predicted_severity_sum"], 0.99))
    return max(row_reference, 1e-6), max(grid_reference, 1e-6)

def compute_feature_importance(model: HistGradientBoostingRegressor, test_frame: pd.DataFrame) -> pd.DataFrame:
    sample_size = min(20_000, len(test_frame))
    sample = test_frame.sample(sample_size, random_state=RANDOM_STATE)
    result = permutation_importance(
        model,
        sample[FEATURE_COLUMNS],
        np.log1p(sample["severity_weight_sum"]),
        n_repeats=3,
        random_state=RANDOM_STATE,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
    )
    importance = pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
        }
    )
    return importance.sort_values("importance_mean", ascending=False).reset_index(drop=True)


## Scenario Layer And Scoring Function

In [6]:
def build_scenario_multipliers(train_crashes: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    alpha = 25.0
    overall_grid = (
        train_crashes.groupby("grid_id", as_index=False)
        .agg(
            overall_crash_count=("CRASH_RECORD_ID", "size"),
            overall_severity_sum=("severity_weight", "sum"),
        )
        .sort_values("grid_id")
    )

    global_overall_crash_count = float(len(train_crashes))
    global_overall_spc = float(train_crashes["severity_weight"].sum() / max(global_overall_crash_count, 1.0))

    masks = {
        "overall": pd.Series(True, index=train_crashes.index),
        "rain": train_crashes["is_rain"] == 1,
        "dark": train_crashes["is_dark"] == 1,
        "wet_surface": train_crashes["is_wet_surface"] == 1,
        "rain_dark_wet": (train_crashes["is_rain"] == 1)
        & (train_crashes["is_dark"] == 1)
        & (train_crashes["is_wet_surface"] == 1),
        "pedbike": train_crashes["is_pedbike"] == 1,
    }

    scenario_parts = []
    global_parts = []
    for scenario_name, mask in masks.items():
        scenario_crashes = train_crashes.loc[mask]
        if scenario_name == "overall":
            scenario_grid = overall_grid.rename(
                columns={
                    "overall_crash_count": "scenario_crash_count",
                    "overall_severity_sum": "scenario_severity_sum",
                }
            ).copy()
            global_scenario_crash_count = global_overall_crash_count
            global_scenario_spc = global_overall_spc
        else:
            scenario_grid = (
                scenario_crashes.groupby("grid_id", as_index=False)
                .agg(
                    scenario_crash_count=("CRASH_RECORD_ID", "size"),
                    scenario_severity_sum=("severity_weight", "sum"),
                )
                .sort_values("grid_id")
            )
            global_scenario_crash_count = float(len(scenario_crashes))
            global_scenario_spc = float(
                scenario_crashes["severity_weight"].sum() / max(global_scenario_crash_count, 1.0)
            )

        merged = overall_grid.merge(scenario_grid, on="grid_id", how="left")
        merged["scenario_name"] = scenario_name
        merged["scenario_crash_count"] = merged["scenario_crash_count"].fillna(0)
        merged["scenario_severity_sum"] = merged["scenario_severity_sum"].fillna(0)
        merged["smoothed_overall_spc"] = (
            merged["overall_severity_sum"] + alpha * global_overall_spc
        ) / (merged["overall_crash_count"] + alpha)
        merged["smoothed_scenario_spc"] = (
            merged["scenario_severity_sum"] + alpha * global_scenario_spc
        ) / (merged["scenario_crash_count"] + alpha)
        merged["scenario_multiplier"] = (
            merged["smoothed_scenario_spc"] / merged["smoothed_overall_spc"].replace({0: np.nan})
        ).clip(0.5, 3.0)
        merged["scenario_multiplier"] = merged["scenario_multiplier"].fillna(1.0)
        scenario_parts.append(merged)

        global_parts.append(
            {
                "scenario_name": scenario_name,
                "global_crash_count": int(global_scenario_crash_count),
                "global_severity_per_crash": float(global_scenario_spc),
                "global_multiplier": float(
                    np.clip(global_scenario_spc / max(global_overall_spc, 1e-9), 0.5, 3.0)
                ),
            }
        )

    scenario_table = pd.concat(scenario_parts, ignore_index=True)
    global_table = pd.DataFrame(global_parts)
    return scenario_table, global_table

def choose_scenario_multiplier(
    grid_id: str,
    scenario_table: pd.DataFrame,
    global_table: pd.DataFrame,
    rain: bool = False,
    dark: bool = False,
    wet_surface: bool = False,
    pedbike: bool = False,
) -> tuple[float, str]:
    scenario_name = "overall"
    if rain and dark and wet_surface:
        scenario_name = "rain_dark_wet"
    elif rain:
        scenario_name = "rain"
    elif dark:
        scenario_name = "dark"
    elif wet_surface:
        scenario_name = "wet_surface"

    global_map = dict(zip(global_table["scenario_name"], global_table["global_multiplier"]))
    multiplier = float(global_map.get(scenario_name, 1.0))

    scenario_subset = scenario_table.loc[
        (scenario_table["grid_id"] == grid_id) & (scenario_table["scenario_name"] == scenario_name),
        "scenario_multiplier",
    ]
    if not scenario_subset.empty:
        multiplier = float(scenario_subset.iloc[0])

    if pedbike:
        ped_multiplier = float(global_map.get("pedbike", 1.0))
        ped_subset = scenario_table.loc[
            (scenario_table["grid_id"] == grid_id) & (scenario_table["scenario_name"] == "pedbike"),
            "scenario_multiplier",
        ]
        if not ped_subset.empty:
            ped_multiplier = float(ped_subset.iloc[0])
        multiplier = float(np.clip(multiplier * ped_multiplier, 0.5, 4.0))
        if scenario_name == "overall":
            scenario_name = "pedbike"
        else:
            scenario_name = f"{scenario_name}_pedbike"

    return multiplier, scenario_name

def score_segment_risk(
    grid_id: str,
    crash_month: int,
    crash_hour: int,
    is_weekend: int,
    feature_tables: dict,
    model: HistGradientBoostingRegressor,
    scenario_table: pd.DataFrame,
    global_scenarios: pd.DataFrame,
    score_reference: float,
    rain: bool = False,
    dark: bool = False,
    wet_surface: bool = False,
    pedbike: bool = False,
) -> dict:
    grid_row = feature_tables["grid_profile"].loc[feature_tables["grid_profile"]["grid_id"] == grid_id]
    if grid_row.empty:
        raise KeyError(f"Unknown grid_id: {grid_id}")

    request = pd.DataFrame(
        {
            "grid_id": [grid_id],
            "grid_center_lat": [float(grid_row["grid_center_lat"].iloc[0])],
            "grid_center_lon": [float(grid_row["grid_center_lon"].iloc[0])],
            "crash_year": [TEST_YEAR],
            "crash_month": [int(crash_month)],
            "crash_hour": [int(crash_hour)],
            "is_weekend": [int(is_weekend)],
            "crash_count": [0],
            "injury_crashes": [0],
            "serious_injury_crashes": [0],
            "fatal_crashes": [0],
            "severity_weight_sum": [0.0],
        }
    )
    request = add_model_features(request, feature_tables)
    base_prediction = float(np.expm1(model.predict(request[FEATURE_COLUMNS]))[0])
    scenario_multiplier, scenario_name = choose_scenario_multiplier(
        grid_id=grid_id,
        scenario_table=scenario_table,
        global_table=global_scenarios,
        rain=rain,
        dark=dark,
        wet_surface=wet_surface,
        pedbike=pedbike,
    )
    adjusted_prediction = base_prediction * scenario_multiplier
    risk_score = float(np.clip(100.0 * adjusted_prediction / max(score_reference, 1e-6), 0.0, 100.0))
    return {
        "grid_id": grid_id,
        "crash_month": int(crash_month),
        "crash_hour": int(crash_hour),
        "is_weekend": int(is_weekend),
        "scenario_name": scenario_name,
        "scenario_multiplier": float(scenario_multiplier),
        "base_prediction": float(base_prediction),
        "adjusted_prediction": float(adjusted_prediction),
        "risk_score_0_100": risk_score,
    }


## Outputs

In [7]:
def write_outputs(
    model: HistGradientBoostingRegressor,
    feature_importance: pd.DataFrame,
    test_frame: pd.DataFrame,
    grid_eval: pd.DataFrame,
    metrics: dict,
    scenario_table: pd.DataFrame,
    global_scenarios: pd.DataFrame,
    sampling_metadata: dict,
    example_score: dict,
) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    model_path = OUTPUT_DIR / "severity_risk_histgb.joblib"
    dump(model, model_path)

    test_output = test_frame[
        [
            "grid_id",
            "grid_center_lat",
            "grid_center_lon",
            "crash_year",
            "crash_month",
            "crash_hour",
            "is_weekend",
            "crash_count",
            "severity_weight_sum",
            "baseline_prediction",
            "model_prediction",
            "risk_score_0_100",
        ]
    ].copy()
    test_output.to_csv(OUTPUT_DIR / "grid_time_predictions_2025.csv.gz", index=False, compression="gzip")
    grid_eval.to_csv(OUTPUT_DIR / "grid_risk_predictions_2025.csv.gz", index=False, compression="gzip")
    scenario_table.to_csv(OUTPUT_DIR / "scenario_multipliers.csv.gz", index=False, compression="gzip")
    global_scenarios.to_csv(OUTPUT_DIR / "scenario_global_multipliers.csv", index=False)
    feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
    grid_eval.nlargest(50, "predicted_severity_sum").to_csv(OUTPUT_DIR / "top_50_risky_grids_2025.csv", index=False)

    metadata = {
        "source_file": str(PREPARED_CRASHES_PATH.relative_to(ROOT)),
        "modeling_window": {"train_start_year": MODELING_START_YEAR, "train_end_year": TRAIN_END_YEAR, "test_year": TEST_YEAR},
        "target": {
            "name": "severity_weight_sum",
            "unit": "grid_id + crash_year + crash_month + crash_hour + is_weekend",
            "description": "Severity-weighted crash sum on a grid-time cell.",
        },
        "model": {
            "type": "HistGradientBoostingRegressor",
            "trained_on": "log1p(severity_weight_sum)",
            "hyperparameters": MODEL_HYPERPARAMETERS,
            "feature_columns": FEATURE_COLUMNS,
            "artifact_path": str(model_path.relative_to(ROOT)),
        },
        "model_settings": {
            "target": "severity_weight_sum on grid_id + crash_year + crash_month + crash_hour + is_weekend",
            "model": "HistGradientBoostingRegressor trained on log1p(severity_weight_sum)",
            "hyperparameters": MODEL_HYPERPARAMETERS,
            "train_test_split": f"{MODELING_START_YEAR}-{TRAIN_END_YEAR} train, {TEST_YEAR} holdout test",
            "zero_row_sampling": sampling_metadata,
            "weights": "Positive grid-time rows use sample_weight=1.0; sampled zero rows use sampled_zero_weight from the full zero-panel expansion.",
            "scenario_multipliers": {
                "scenarios": SCENARIO_NAMES,
                "method": "Grid-level smoothed severity-per-crash ratios with alpha=25, clipped to 0.5-3.0; pedbike can be combined with weather/light/surface scenarios and is clipped to 0.5-4.0.",
            },
        },
        "sampling": sampling_metadata,
        "metrics": metrics,
        "example_score": example_score,
        "output_files": {
            "grid_time_predictions": str((OUTPUT_DIR / "grid_time_predictions_2025.csv.gz").relative_to(ROOT)),
            "grid_predictions": str((OUTPUT_DIR / "grid_risk_predictions_2025.csv.gz").relative_to(ROOT)),
            "scenario_multipliers": str((OUTPUT_DIR / "scenario_multipliers.csv.gz").relative_to(ROOT)),
            "global_scenarios": str((OUTPUT_DIR / "scenario_global_multipliers.csv").relative_to(ROOT)),
            "feature_importance": str((OUTPUT_DIR / "feature_importance.csv").relative_to(ROOT)),
            "top_grids": str((OUTPUT_DIR / "top_50_risky_grids_2025.csv").relative_to(ROOT)),
        },
    }
    (OUTPUT_DIR / "model_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    (OUTPUT_DIR / "modeling_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

def main() -> dict:
    crashes = load_prepared_crashes()
    grid_lookup = build_grid_lookup(crashes)
    positive_panel = build_positive_panel(crashes, grid_lookup)

    train_crashes = crashes[crashes["crash_year"].between(MODELING_START_YEAR, TRAIN_END_YEAR)].copy()
    train_positive = positive_panel[positive_panel["crash_year"].between(MODELING_START_YEAR, TRAIN_END_YEAR)].copy()
    test_positive = positive_panel[positive_panel["crash_year"] == TEST_YEAR].copy()

    feature_tables = build_feature_tables(train_crashes, train_positive, grid_lookup)
    sampled_zero, sampling_metadata = sample_training_zero_panel(train_positive, grid_lookup)

    positive_train = train_positive.copy()
    positive_train["sample_weight"] = 1.0
    train_frame = pd.concat([positive_train, sampled_zero], ignore_index=True)
    train_frame = add_model_features(train_frame, feature_tables)

    test_frame = build_full_test_panel(grid_lookup, test_positive)
    test_frame = add_model_features(test_frame, feature_tables)
    test_frame["baseline_prediction"] = test_frame["grid_time_mean_target"].clip(lower=0)

    model = train_model(train_frame)
    predicted_log = model.predict(test_frame[FEATURE_COLUMNS])
    test_frame["model_prediction"] = np.expm1(predicted_log).clip(min=0)

    metrics, grid_eval = evaluate_predictions(test_frame)
    row_reference, grid_reference = build_prediction_scores(test_frame, grid_eval)
    test_frame["risk_score_0_100"] = np.clip(100.0 * test_frame["model_prediction"] / row_reference, 0.0, 100.0)
    grid_eval["grid_risk_score_0_100"] = np.clip(100.0 * grid_eval["predicted_severity_sum"] / grid_reference, 0.0, 100.0)

    feature_importance = compute_feature_importance(model, test_frame)
    scenario_table, global_scenarios = build_scenario_multipliers(train_crashes)

    top_grid_id = str(grid_eval.iloc[0]["grid_id"])
    example_score = score_segment_risk(
        grid_id=top_grid_id,
        crash_month=11,
        crash_hour=18,
        is_weekend=0,
        feature_tables=feature_tables,
        model=model,
        scenario_table=scenario_table,
        global_scenarios=global_scenarios,
        score_reference=row_reference,
        rain=True,
        dark=True,
        wet_surface=True,
    )

    write_outputs(
        model=model,
        feature_importance=feature_importance,
        test_frame=test_frame,
        grid_eval=grid_eval,
        metrics=metrics,
        scenario_table=scenario_table,
        global_scenarios=global_scenarios,
        sampling_metadata=sampling_metadata,
        example_score=example_score,
    )

    summary = {
        "crash_rows": int(len(crashes)),
        "train_positive_rows": int(len(train_positive)),
        "test_rows": int(len(test_frame)),
        "unique_grids": int(len(grid_lookup)),
        "row_rmse": float(metrics["row_level"]["model_rmse"]),
        "grid_rmse": float(metrics["grid_level"]["model_rmse"]),
        "top_grid_overlap": float(metrics["grid_level"]["top_100_overlap_model"]),
        "example_score": example_score,
    }
    print(json.dumps(summary, indent=2))
    return summary


## Run Pipeline

In [8]:
summary = main()
summary

{
  "crash_rows": 870705,
  "train_positive_rows": 661398,
  "test_rows": 1234944,
  "unique_grids": 2144,
  "row_rmse": 0.4435487176834949,
  "grid_rmse": 19.67980000665151,
  "top_grid_overlap": 0.69,
  "example_score": {
    "grid_id": "58_52",
    "crash_month": 11,
    "crash_hour": 18,
    "is_weekend": 0,
    "scenario_name": "rain_dark_wet",
    "scenario_multiplier": 1.5730688300576896,
    "base_prediction": 0.4939529345285034,
    "adjusted_prediction": 0.7770219648223154,
    "risk_score_0_100": 100.0
  }
}


{'crash_rows': 870705,
 'train_positive_rows': 661398,
 'test_rows': 1234944,
 'unique_grids': 2144,
 'row_rmse': 0.4435487176834949,
 'grid_rmse': 19.67980000665151,
 'top_grid_overlap': 0.69,
 'example_score': {'grid_id': '58_52',
  'crash_month': 11,
  'crash_hour': 18,
  'is_weekend': 0,
  'scenario_name': 'rain_dark_wet',
  'scenario_multiplier': 1.5730688300576896,
  'base_prediction': 0.4939529345285034,
  'adjusted_prediction': 0.7770219648223154,
  'risk_score_0_100': 100.0}}

## Model Settings

This table records the target, estimator, hyperparameters, train/test split, zero-row sampling, weights, and scenario multiplier settings used by the executed Phase 4 pipeline.

In [9]:
modeling_metadata = json.loads((OUTPUT_DIR / "modeling_metadata.json").read_text())
settings = modeling_metadata["model_settings"]
sampling = settings["zero_row_sampling"]

model_settings_table = pd.DataFrame(
    [
        {"setting": "target", "value": settings["target"]},
        {"setting": "model", "value": settings["model"]},
        {
            "setting": "hyperparameters",
            "value": "; ".join(f"{key}={value}" for key, value in settings["hyperparameters"].items()),
        },
        {"setting": "train/test split", "value": settings["train_test_split"]},
        {
            "setting": "zero-row sampling",
            "value": (
                f"train panel rows={sampling['train_panel_total_rows']:,}; "
                f"positive rows={sampling['train_positive_rows']:,}; "
                f"zero rows={sampling['train_zero_rows_total']:,}; "
                f"sampled zeros={sampling['train_zero_rows_sampled']:,}"
            ),
        },
        {
            "setting": "weights",
            "value": f"positive rows=1.0; sampled zero rows={sampling['sampled_zero_weight']:.5f}",
        },
        {
            "setting": "scenario multipliers",
            "value": (
                f"scenarios={', '.join(settings['scenario_multipliers']['scenarios'])}; "
                f"{settings['scenario_multipliers']['method']}"
            ),
        },
    ]
)
display(model_settings_table)

,setting,value
0,target,severity_weight_sum on grid_id + crash_year + ...
1,model,HistGradientBoostingRegressor trained on log1p...
2,hyperparameters,loss=squared_error; learning_rate=0.05; max_it...
3,train/test split,"2018-2024 train, 2025 holdout test"
4,zero-row sampling,"train panel rows=8,644,608; positive rows=661,..."
5,weights,positive rows=1.0; sampled zero rows=10.64428
6,scenario multipliers,"scenarios=overall, rain, dark, wet_surface, ra..."
